In [ ]:
import math


In [ ]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,1))
X = torch.rand(size=(2,4))

net(X)

In [ ]:
print(net[2].state_dict())
print(net)
print(X)

手写self-attention

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, dim) -> None:
        super().__init__()
        self.dim = dim

        # 这样很清晰
        self.query_proj = nn.Linear(dim, dim)
        self.key_proj = nn.Linear(dim, dim)
        self.value_proj = nn.Linear(dim, dim)
        # 一般是 0.1 的 dropout，一般写作 config.attention_probs_dropout_prob
        # hidden_dropout_prob 一般也是 0.1
        self.att_drop = nn.Dropout(0.1)

        # 可以不写；具体和面试官沟通。
        # 这是 MultiHeadAttention 中的产物，这个留给 MultiHeadAttention 也没有问题；
        self.output_proj = nn.Linear(dim, dim)

    def forward(self, X, attention_mask=None):
        # attention_mask shape is: (batch, seq)
        # X shape is: (batch, seq, dim)

        Q = self.query_proj(X)
        K = self.key_proj(X)
        V = self.value_proj(X)

        att_weight = Q @ K.transpose(-1, -2) / math.sqrt(self.dim)
        if attention_mask is not None:
            # 给 weight 填充一个极小的值
            att_weight = att_weight.masked_fill(attention_mask == 0, float("-1e20"))

        att_weight = torch.softmax(att_weight, dim=-1)
        print(att_weight)

        # 这里在 BERT中的官方代码也说很奇怪，但是原文中这么用了，所以继承了下来
        # （用于 output 后面会更符合直觉？）
        att_weight = self.att_drop(att_weight)

        output = att_weight @ V
        ret = self.output_proj(output)
        return ret

手写MHA

In [ ]:
class MultiSelfAttention(nn.Module):
    
    def __init__(self, hidden_dim: int, nums_head: int, dropout_rate: float = 0.1):
            super().__init__()
            self.nums_head = nums_head
            self.nums_dim = hidden_dim // nums_head
            
            self.query_proj = nn.Linear(hidden_dim, hidden_dim)
            self.key_proj = nn.Linear(hidden_dim, hidden_dim)
            self.value_proj = nn.Linear(hidden_dim, hidden_dim)
            
            self.dropout = nn.Dropout(dropout_rate)
            self.o_proj = nn.Linear(hidden_dim, hidden_dim)
    def forward(self, X, attention_mask=None):
        batch_size, seq_size, _ = X.size()
        Q = self.query_proj(X)
        K = self.key_proj(X)
        V = self.value_proj(X)
        
        Q = Q.view(batch_size, seq_size, self.nums_head, self.nums_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, seq_size, self.nums_head, self.nums_dim).transpose(1, 2)
        V = V.view(batch_size, seq_size, self.nums_head, self.nums_dim).transpose(1, 2)
        
        att_weight = Q @ K.transpose(-1, -2) / math.sqrt(self.nums_dim)
        if attention_mask is not None:
            # 给 weight 填充一个极小的值
            att_weight = att_weight.masked_fill(attention_mask == 0, float("-1e20"))

        att_weight = torch.softmax(att_weight, dim=-1)
        print(att_weight)

        # 这里在 BERT中的官方代码也说很奇怪，但是原文中这么用了，所以继承了下来
        # （用于 output 后面会更符合直觉？）

        output = att_weight @ V
        output = output.transpose(1, 2).contiguous()
        ## 内存连续化，要不然不能用view
        output = output.view(batch_size, seq_size, -1) ## -1 自动填充h
        ret = self.o_proj(output)
        return ret


In [55]:
### MHA测试

X = torch.rand(3, 2, 128)
mark = torch.tensor(
    [
        [0, 1],
        [0,0],
        [1,0]
    ]
)
mark = mark.unsqueeze(1).unsqueeze(2).expand(3, 8, 2, 2)
net = MultiSelfAttention(128, 8)
net(X, attention_mask = mark)

tensor([[[[0.0000, 1.0000],
          [0.0000, 1.0000]],

         [[0.0000, 1.0000],
          [0.0000, 1.0000]],

         [[0.0000, 1.0000],
          [0.0000, 1.0000]],

         [[0.0000, 1.0000],
          [0.0000, 1.0000]],

         [[0.0000, 1.0000],
          [0.0000, 1.0000]],

         [[0.0000, 1.0000],
          [0.0000, 1.0000]],

         [[0.0000, 1.0000],
          [0.0000, 1.0000]],

         [[0.0000, 1.0000],
          [0.0000, 1.0000]]],


        [[[0.5000, 0.5000],
          [0.5000, 0.5000]],

         [[0.5000, 0.5000],
          [0.5000, 0.5000]],

         [[0.5000, 0.5000],
          [0.5000, 0.5000]],

         [[0.5000, 0.5000],
          [0.5000, 0.5000]],

         [[0.5000, 0.5000],
          [0.5000, 0.5000]],

         [[0.5000, 0.5000],
          [0.5000, 0.5000]],

         [[0.5000, 0.5000],
          [0.5000, 0.5000]],

         [[0.5000, 0.5000],
          [0.5000, 0.5000]]],


        [[[1.0000, 0.0000],
          [1.0000, 0.0000]],

         [

tensor([[[ 2.4033e-01,  2.0993e-01,  5.8890e-02,  8.7040e-02, -6.3862e-02,
           3.7180e-03,  6.2651e-02, -1.3903e-01,  7.3345e-02,  8.5441e-02,
           2.0933e-01,  1.5505e-01, -1.6602e-01, -6.0647e-02, -8.3757e-02,
           7.0581e-02,  3.3743e-02, -3.8020e-02, -3.0245e-01,  2.8189e-02,
           1.5751e-02,  1.6334e-01,  2.1307e-02, -8.4882e-02,  3.3636e-01,
          -1.0279e-01,  1.5418e-02,  3.3842e-01,  9.6794e-02, -5.5669e-02,
          -1.6299e-01, -4.3538e-01, -1.5580e-01,  1.1184e-01,  3.1001e-02,
          -5.4529e-02, -3.0732e-01, -1.0795e-02, -5.4972e-02,  2.3968e-01,
          -2.6610e-01, -2.8287e-01, -2.0579e-01, -9.9586e-02, -1.3772e-01,
           1.2677e-01,  1.1608e-01, -7.0981e-03,  3.2958e-01, -1.7923e-01,
           1.3722e-01,  8.9763e-02,  4.2324e-02,  2.6392e-01, -4.5218e-02,
           1.0682e-02, -2.8498e-01, -8.5294e-02,  3.4906e-01,  3.3391e-01,
          -4.1453e-03,  1.0487e-01,  2.4511e-01,  8.5698e-02, -1.6586e-01,
           1.5403e-01, -2

In [ ]:
### 测试

X = torch.rand(3, 4, 2)
mark = torch.tensor(
    [
        [1,1,1,0],
        [1,1,0,0],
        [1,0,0,0]
    ]
)
mark = mark.unsqueeze(dim = 1).repeat(1, 4, 1)
net = SelfAttention(2)
net(X, attention_mask = mark)